# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print("Version:", metadata.version)
print("Identifier:", metadata.identifier)
print("Published Date:", metadata.datePublished)
print("License:", metadata.license)
print("Authors (@id):", [a['@id'] for a in metadata.author])

## 2. Data Overview
Review available record sets and their fields. All entities are referenced by their `@id`.

**Note**: The FAIR^2 schema includes tabular clinical data. Let's list the available record sets and review the structure.

In [ ]:
# List available record sets and their @id

record_sets = dataset.record_sets
print("Available record sets:")
for rs in record_sets:
    print(f"- Name: {rs.name} | @id: {rs['@id']}")

# For demonstration, show the fields and columns for each record set
for rs in record_sets:
    print(f"\nRecord Set: {rs.name} (@id: {rs['@id']})")
    fields = rs.fields
    print("Fields:")
    for f in fields:
        print(f"  - {f['name']} (@id: {f['@id']}) | Type: {f['dataType']}")
    columns = rs.columns if hasattr(rs, 'columns') else []
    if columns:
        print("Columns:")
        for c in columns:
            print(f"  - {c['name']} (@id: {c['@id']})")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.

Use the record set and field `@id`s identified above.

In [ ]:
# Extract data from all available record sets (by @id)
record_set_ids = [rs['@id'] for rs in dataset.record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if len(records) == 0:
        print(f"Record set @{record_set_id} yielded no records.")
        continue
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nColumns for {record_set_id}:")
    print(df.columns.tolist())
    print(df.head())

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, or grouping data by key attributes. All entities are referenced using their `@id`.

In [ ]:
# Example EDA: Filter, Normalize, Group
# Choose a record set and numeric field for demonstration

if dataframes:
    # Select the first available record set and try to find a numeric field
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    
    # Attempt to auto-detect a numeric field
    numeric_field_candidates = [col for col in df.columns if df[col].dtype in ['int64', 'float64']]
    if numeric_field_candidates:
        numeric_field = numeric_field_candidates[0]
        # Filter by threshold
        threshold = df[numeric_field].mean()  # Use mean as threshold for demo
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"].head()])

        # Try to group by a categorical field
        group_field_candidates = [col for col in df.columns if df[col].dtype == 'object']
        if group_field_candidates:
            group_field = group_field_candidates[0]
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field}:")
            print(grouped_df.head())
    else:
        print("No numeric fields detected for EDA.")
else:
    print("No record sets with data; EDA cannot be performed.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset.

Let's plot a histogram for the selected numeric field and a barplot for the grouped data, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and len(dataframes) > 0:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    numeric_field_candidates = [col for col in df.columns if df[col].dtype in ['int64', 'float64']]
    if numeric_field_candidates:
        numeric_field = numeric_field_candidates[0]
        plt.figure(figsize=(8, 6))
        sns.histplot(df[numeric_field], bins=15, kde=True)
        plt.title(f"Distribution of {numeric_field} in record set @{record_set_id}")
        plt.xlabel(numeric_field)
        plt.ylabel("Count")
        plt.show()
    else:
        print("No numeric field for visualization.")
    # Barplot if grouped data exists
    group_field_candidates = [col for col in df.columns if df[col].dtype == 'object']
    if numeric_field_candidates and group_field_candidates:
        group_field = group_field_candidates[0]
        plt.figure(figsize=(8, 6))
        df_grouped = df.groupby(group_field)[numeric_field].mean()
        df_grouped.plot(kind='bar')
        plt.title(f"Mean {numeric_field} grouped by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion

In this notebook, we loaded the FAIR^2 colorectal cancer survivors dataset using the Croissant schema and `mlcroissant`. We inspected metadata, reviewed available record sets and fields referenced by `@id`, extracted tabular data, performed basic filtering and normalization, and visualized numeric distributions and categorical groupings.

Further analysis can be performed by focusing on specific clinical or molecular attributes, stratification by MSI-H status, or extending preprocessing for predictive modeling. The FAIR^2 schema ensures robust data referencing and reproducibility for downstream research.